In [1]:
import pyranges1 as pr
import pandas as pd
import subprocess
from pathlib import Path
import gzip
import pysam
from contextlib import nullcontext

In [2]:
pd.set_option('display.max_columns', None)

## GFF preparation + BED preparation

In [3]:
gtf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz'

In [4]:
command = f'''./prepare_data -i {gtf_filepath} -rm 1 --add_chr'''
subprocess.run(command, shell=True, check=True)

CompletedProcess(args='./prepare_data -i /Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.gtf.gz -rm 1 --add_chr', returncode=0)

## MNV calculation from VCF

In [ ]:
input_vcf_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.vcf.gz'
fasta_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa'
bam_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/test_files/bam/000007239270.bam.genozip'
reference_genozip_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.ref.genozip'
bed_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/cds.bed'
cds_table_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/cds_table.parquet'
output_vcf_filepath = input_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.mnv.vcf.gz'

command = f'''./cis2mnv -i {input_vcf_filepath} \
        --fasta {fasta_filepath} \
        --bam {bam_genozip_filepath} \
        --genozip_ref {reference_genozip_filepath} \
        --bed {bed_filepath} \
        --merge_overlapped_regions 0 \
        --only_pass_variants 1 \
        --cds_table {cds_table_filepath} \
        --remove_intermediate_files 1 \
        --compress 1 \
        -o {output_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)

## VCF annotation

### VEP

In [ ]:
gtf_single_transcripts_filepath = '/Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.main_transcripts.gtf.gz'
vep_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.vep.vcf.gz'

command = f'''mamba run -n cis2mnv-vep.venv vep \
    --input_file {output_vcf_filepath} \
    --output_file {vep_annotated_vcf_filepath} \
    --gtf {gtf_single_transcripts_filepath} \
    --fasta {fasta_filepath} \
    --warning_file STDERR \
    --no_stats \
    --force_overwrite \
    --format vcf \
    --vcf \
    --compress_output bgzip \
    --symbol \
    --protein \
    --hgvs'''
subprocess.run(command, shell=True, check=True)

CompletedProcess(args='mamba run -n cis2mnv-vep.venv vep     --input_file /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.mnv.vcf.gz     --output_file /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.mnv.vep.vcf.gz     --gtf /Users/andrejnekrasov/pro/my/cisfuse/data/refs/Homo_sapiens.GRCh38.116.chr.main_transcripts.gtf.gz     --fasta /Users/andrejnekrasov/pro/my/cisfuse/data/refs/GRCh38.d1.vd1.fa     --warning_file STDERR     --no_stats     --force_overwrite     --format vcf     --vcf     --compress_output bgzip     --symbol     --protein     --hgvs', returncode=0)

### SnpEff

In [ ]:
snpeff_annotated_vcf_filepath = output_vcf_filepath.removesuffix('.gz').removesuffix('.vcf') + '.snpeff.vcf.gz'

command = f'''mamba run -n cis2mnv-snpeff.venv snpeff ann \
    -canon \
    -noStats \
    -download hg38 {output_vcf_filepath} \
    | bgzip -c > {snpeff_annotated_vcf_filepath}'''
subprocess.run(command, shell=True, check=True)

CompletedProcess(args='mamba run -n cis2mnv-snpeff.venv snpeff ann     -canon     -noStats     -download hg38 /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.mnv.vcf.gz     | bgzip -c > /Users/andrejnekrasov/pro/my/cisfuse/data/test_files/vcf/000007239270_DeepVariant.mnv.snpeff.vcf.gz', returncode=0)